# Financial Statement ETL Validation

This notebook checks PostgreSQL coverage, duplicate candidates, calculated values, TTM totals, and selected values against KAP filings. Internal checks can reveal inconsistencies; comparisons with original filings are needed to confirm source accuracy.

## 1. Scope and connection

Set the symbols and period below. Connection details come from BIST_FINANCE_DSN or DATABASE_URL. The helper verifies the database is bist_finance. Run the notebook with the project root as the working directory.

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display
from plotly.colors import qualitative
from plotly.subplots import make_subplots

from services.metadata_store import connect_metadata_db

SYMBOLS = ["THYAO", "TUPRS", "BIMAS", "ASELS", "EREGL", "SISE", "FROTO", "TCELL", "MGROS", "ENKAI"]
BEGIN_YEAR, BEGIN_QUARTER = 2017, 1
END_YEAR, END_QUARTER = 2026, 2
STATEMENT_TYPES = ["income_statement", "balance_sheet", "cash_flow", "equity_change"]

def read_query(sql, params=()):
    with connect_metadata_db() as conn:
        rows = conn.execute(sql, params).fetchall()
    return pd.DataFrame(rows)

with connect_metadata_db() as conn:
    print(f"Connected to PostgreSQL database: {conn.info.dbname}")
print(f"Reviewing {len(SYMBOLS)} companies, {BEGIN_YEAR} Q{BEGIN_QUARTER} through {END_YEAR} Q{END_QUARTER}")

## 2. Period coverage

Missing periods are review prompts, not automatic errors: a company may have listed later or may not have a filing in the selected range.

In [ ]:
start_period = BEGIN_YEAR * 4 + BEGIN_QUARTER
end_period = END_YEAR * 4 + END_QUARTER
periods = [((p - 1) // 4, (p - 1) % 4 + 1) for p in range(start_period, end_period + 1)]
expected = {(symbol, year, quarter) for symbol in SYMBOLS for year, quarter in periods}
found = read_query(
    """SELECT c.symbol, s.statement_type, s.fiscal_year, s.fiscal_quarter
       FROM silver.current_financial_observations s
       JOIN metadata.companies c USING (company_id)
       WHERE c.symbol = ANY(%s)
         AND (s.fiscal_year * 4 + s.fiscal_quarter) BETWEEN %s AND %s
       GROUP BY c.symbol, s.statement_type, s.fiscal_year, s.fiscal_quarter""",
    (SYMBOLS, start_period, end_period),
)
summary, missing = [], []
for statement_type in STATEMENT_TYPES:
    rows = found.loc[found["statement_type"] == statement_type, ["symbol", "fiscal_year", "fiscal_quarter"]]
    existing = set(rows.itertuples(index=False, name=None))
    missing_pairs = expected - existing
    summary.append({"statement_type": statement_type, "expected_company_quarters": len(expected),
                    "existing_company_quarters": len(existing & expected), "missing_company_quarters": len(missing_pairs),
                    "coverage_pct": round(100 * (len(expected) - len(missing_pairs)) / len(expected), 1)})
    missing.extend({"statement_type": statement_type, "symbol": symbol, "year": year, "quarter": quarter}
                   for symbol, year, quarter in sorted(missing_pairs))
display(pd.DataFrame(summary))
display(pd.DataFrame(missing))

## 3. Duplicate candidates

Rows are grouped by company, period, statement type, hierarchy path, item, and source column. Zero amounts are excluded. A candidate still needs source-file review before being treated as an error.

In [ ]:
duplicate_candidates = read_query(
    """SELECT c.symbol, s.statement_type, s.fiscal_year, s.fiscal_quarter,
              s.line_item_path, s.item, s.source_column_id, s.column_label,
              COUNT(*) AS row_count, ARRAY_AGG(DISTINCT f.source_filename) AS source_files
       FROM silver.current_financial_observations s
       JOIN metadata.companies c USING (company_id)
       JOIN bronze.statement_filings f USING (filing_key)
       WHERE c.symbol = ANY(%s) AND s.amount <> 0
       GROUP BY c.symbol, s.statement_type, s.fiscal_year, s.fiscal_quarter,
                s.line_item_path, s.item, s.source_column_id, s.column_label
       HAVING COUNT(*) > 1
       ORDER BY row_count DESC, c.symbol, s.fiscal_year, s.fiscal_quarter""",
    (SYMBOLS,),
)
print(f"Duplicate candidate groups excluding zero amounts: {len(duplicate_candidates)}")
display(duplicate_candidates.head(100))

## 4. Inflation and currency calculations

Recalculate expected real values from monthly CPI and each Gold row's target month, then check USD values using the quarter-end monthly USD/TRY rate.

In [ ]:
adjustments = read_query(
    """WITH cpi_index AS (
           SELECT observation_date, EXP(SUM(LN(1 + value / 100.0)) OVER (ORDER BY observation_date)) AS price_index
           FROM silver.monthly_macro_data WHERE series_code = 'CPI_MONTHLY_RATE'
       )
       SELECT c.symbol, g.statement_type, g.fiscal_year, g.fiscal_quarter, g.item,
              g.value_quarter, g.value_real, g.value_usd,
              source_cpi.price_index AS source_price_index,
              target_cpi.price_index AS target_price_index, fx.value AS usd_try
       FROM gold.current_financial_values g
       JOIN metadata.companies c USING (company_id)
       LEFT JOIN cpi_index source_cpi ON source_cpi.observation_date = make_date(g.fiscal_year, g.fiscal_quarter * 3, 1)
       LEFT JOIN cpi_index target_cpi ON target_cpi.observation_date = make_date(g.inflation_base_year, g.inflation_base_month, 1)
       LEFT JOIN silver.monthly_macro_data fx ON fx.series_code = 'USDTRY_MONTHLY_OPEN'
            AND fx.observation_date = make_date(g.fiscal_year, g.fiscal_quarter * 3, 1)
       WHERE c.symbol = ANY(%s)""",
    (SYMBOLS,),
)
for column in ["value_quarter", "value_real", "value_usd", "source_price_index", "target_price_index", "usd_try"]:
    adjustments[column] = pd.to_numeric(adjustments[column], errors="coerce")
adjustments["expected_real"] = adjustments["value_quarter"] * adjustments["target_price_index"] / adjustments["source_price_index"]
adjustments["expected_usd"] = adjustments["value_quarter"] / adjustments["usd_try"]
real_ok = np.isclose(adjustments["value_real"], adjustments["expected_real"], rtol=1e-8, atol=1e-5, equal_nan=True)
usd_ok = np.isclose(adjustments["value_usd"], adjustments["expected_usd"], rtol=1e-8, atol=1e-5, equal_nan=True)
display(pd.DataFrame([{"rows": len(adjustments),
    "missing_source_or_target_cpi": int(adjustments[["source_price_index", "target_price_index"]].isna().any(axis=1).sum()),
    "real_value_mismatches": int((~real_ok).sum()), "missing_usd_try": int(adjustments["usd_try"].isna().sum()),
    "usd_value_mismatches": int((~usd_ok).sum())}]))
display(adjustments.loc[~(real_ok & usd_ok)].head(30))

## 5. TTM consistency

For rows with four available quarters, compare stored TTM totals with the sum of the corresponding four quarter values.

In [ ]:
ttm_check = read_query(
    """WITH expected AS (
           SELECT t.financial_value_id, COUNT(p.value_quarter) AS period_count,
                  SUM(p.value_quarter) AS expected_ttm, SUM(p.value_real) AS expected_real_ttm
           FROM gold.current_financial_values t
           LEFT JOIN gold.current_financial_values p
             ON p.company_id = t.company_id AND p.statement_type = t.statement_type
            AND p.line_item_path = t.line_item_path AND p.item = t.item
            AND p.source_column_id IS NOT DISTINCT FROM t.source_column_id
            AND (p.fiscal_year * 4 + p.fiscal_quarter) BETWEEN (t.fiscal_year * 4 + t.fiscal_quarter - 3)
                AND (t.fiscal_year * 4 + t.fiscal_quarter)
           WHERE t.statement_type = 'income_statement'
             AND t.company_id IN (SELECT company_id FROM metadata.companies WHERE symbol = ANY(%s))
           GROUP BY t.financial_value_id
       )
       SELECT c.symbol, g.fiscal_year, g.fiscal_quarter, g.line_item_path, g.item, e.period_count,
              g.value_ttm, e.expected_ttm, g.value_real_ttm, e.expected_real_ttm
       FROM gold.current_financial_values g
       JOIN expected e USING (financial_value_id)
       JOIN metadata.companies c USING (company_id)
       WHERE e.period_count = 4 AND g.statement_type = 'income_statement'
       ORDER BY c.symbol, g.fiscal_year, g.fiscal_quarter, g.line_item_path""",
    (SYMBOLS,),
)
for column in ["value_ttm", "expected_ttm", "value_real_ttm", "expected_real_ttm"]:
    ttm_check[column] = pd.to_numeric(ttm_check[column], errors="coerce")
ttm_check["nominal_matches"] = np.isclose(ttm_check["value_ttm"], ttm_check["expected_ttm"], rtol=1e-8, atol=1e-5, equal_nan=True)
ttm_check["real_matches"] = np.isclose(ttm_check["value_real_ttm"], ttm_check["expected_real_ttm"], rtol=1e-8, atol=1e-5, equal_nan=True)
print(f"Checked {len(ttm_check)} rows; nominal mismatches: {(~ttm_check['nominal_matches']).sum()}; real mismatches: {(~ttm_check['real_matches']).sum()}")
display(ttm_check.loc[~(ttm_check["nominal_matches"] & ttm_check["real_matches"])].head(30))

## 6. Gross-margin formula spot check

Recompute quarterly gross margin from revenue and cost of sales, then compare it with the Gold ratio.

In [ ]:
gross_margin_check = read_query(
    """WITH source_values AS (
           SELECT g.company_id, g.fiscal_year, g.fiscal_quarter,
                  SUM(g.value_quarter) FILTER (WHERE g.item = 'hasılat') AS revenue,
                  SUM(g.value_quarter) FILTER (WHERE g.item = 'satışların maliyeti') AS cost_of_sales
           FROM gold.current_financial_values g
           WHERE g.statement_type = 'income_statement' AND g.line_item_path LIKE 'profit or loss >%'
             AND g.company_id IN (SELECT company_id FROM metadata.companies WHERE symbol = ANY(%s))
           GROUP BY g.company_id, g.fiscal_year, g.fiscal_quarter
       )
       SELECT c.symbol, s.fiscal_year, s.fiscal_quarter, r.value_quarter AS stored_gross_margin,
              (s.revenue - ABS(s.cost_of_sales)) / NULLIF(s.revenue, 0) AS expected_gross_margin
       FROM source_values s JOIN metadata.companies c USING (company_id)
       LEFT JOIN gold.current_financial_ratios r ON r.company_id = s.company_id
            AND r.fiscal_year = s.fiscal_year AND r.fiscal_quarter = s.fiscal_quarter
            AND r.ratio_code = 'gross_margin'
       ORDER BY c.symbol, s.fiscal_year, s.fiscal_quarter""",
    (SYMBOLS,),
)
gross_margin_check["matches"] = np.isclose(
    pd.to_numeric(gross_margin_check["stored_gross_margin"], errors="coerce"),
    pd.to_numeric(gross_margin_check["expected_gross_margin"], errors="coerce"), rtol=1e-8, atol=1e-8, equal_nan=True,
)
print(f"Gross-margin rows checked: {len(gross_margin_check)}; mismatches: {(~gross_margin_check['matches']).sum()}")
display(gross_margin_check.loc[~gross_margin_check["matches"]].head(30))

## 7. Compare against a KAP filing

Enter cumulative amounts transcribed from the filing. For Q1, use zero as the prior cumulative value. The comparison calculates a standalone quarter before comparing it with Gold.

In [ ]:
manual_source_values = pd.DataFrame([{
    "symbol": "THYAO", "fiscal_year": 2025, "fiscal_quarter": 4, "item": "hasılat",
    "kap_cumulative_value": 955472000000, "prior_cumulative_value": 690825000000,
    "source_note": "Add KAP filing URL or report ID",
}])
database_values = read_query(
    """SELECT c.symbol, g.fiscal_year, g.fiscal_quarter, g.line_item_path, g.item,
              g.value_quarter, g.value_real, g.value_usd, g.column_label, g.source_column_id
       FROM gold.current_financial_values g JOIN metadata.companies c USING (company_id)
       WHERE c.symbol = ANY(%s)""",
    (SYMBOLS,),
)
source_comparison = manual_source_values.merge(
    database_values, on=["symbol", "fiscal_year", "fiscal_quarter", "item"], how="left"
)
source_comparison["kap_standalone_quarter_value"] = source_comparison["kap_cumulative_value"] - source_comparison["prior_cumulative_value"]
source_comparison["difference"] = source_comparison["value_quarter"] - source_comparison["kap_standalone_quarter_value"]
source_comparison["absolute_difference"] = source_comparison["difference"].abs()
display(source_comparison)

## 8. Quarterly financial trend

Choose a company to compare nominal and inflation-adjusted quarterly revenue and net profit.

In [ ]:
PLOT_SYMBOL = "MGROS"
plotted = read_query(
    """SELECT g.fiscal_year AS year, g.fiscal_quarter AS quarter, g.item,
              g.value_quarter AS value, g.value_real
       FROM gold.current_financial_values g JOIN metadata.companies c USING (company_id)
       WHERE c.symbol = %s AND g.statement_type = 'income_statement'
         AND g.item IN ('hasılat', 'dönem karı (zararı)')
       ORDER BY g.fiscal_year, g.fiscal_quarter""",
    (PLOT_SYMBOL,),
)
plotted["period"] = plotted["year"].astype(str) + " Q" + plotted["quarter"].astype(str)
period_order = plotted["period"].drop_duplicates().tolist()
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.12,
    subplot_titles=(f"{PLOT_SYMBOL}: nominal quarterly values", f"{PLOT_SYMBOL}: inflation-adjusted quarterly values"))
colors = qualitative.Plotly
for index, (item, group) in enumerate(plotted.groupby("item", sort=True)):
    color = colors[index % len(colors)]
    for row, column in [(1, "value"), (2, "value_real")]:
        fig.add_trace(go.Scatter(x=group["period"], y=group[column] / 1e9, mode="lines+markers",
            name=item, legendgroup=item, showlegend=row == 1, line={"color": color},
            marker={"color": color, "size": 7}, hovertemplate=f"%{{x}}<br>%{{y:.2f}} TRY bn<extra>{item}</extra>"), row=row, col=1)
fig.update_xaxes(categoryorder="array", categoryarray=period_order, tickangle=70, row=2, col=1)
fig.update_yaxes(title_text="TRY billions", row=1, col=1)
fig.update_yaxes(title_text="TRY billions", row=2, col=1)
fig.update_layout(height=800, width=1300, template="plotly_white", legend_title_text="Item", hovermode="x unified")
fig.show()

## 9. KAP statement screenshots

Add original statement screenshots here with company, period, statement type, and filing link or ID.

### Screenshot 1


### Screenshot 2


### Screenshot 3